In [1]:
# ==============================================================
# CELL 1 — CONFIG
# ==============================================================
import numpy as np
import laspy
from pathlib import Path

scan_a_path = "/media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/merged/ALL/merged_7000_VUX_PUCK.las"
scan_b_path = "/media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/merged/ALL/merged_8000_VUX_PUCK.las"

# fenêtre temporelle à extraire sur chaque ligne de scan
# t_crop_a = (315976, 315987.5)
# t_crop_b = (315988, 316030)

t_crop_a = (315845, 315860)
t_crop_b = (315915, 315927)

out_dir = Path("/media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/limatch_debug/7000_8000")
out_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# ==============================================================
# CELL 2 — CROP ET SAVE
# ==============================================================
def crop_las_by_time(las_path, t0, t1, out_path):
    las = laspy.read(las_path)
    t = np.asarray(las.gps_time)
    mask = (t >= t0) & (t <= t1)
    n_total = len(t)
    n_kept  = int(mask.sum())
    print(f"{Path(las_path).name} : {n_total} pts → {n_kept} pts dans [{t0:.3f}, {t1:.3f}]")
    if n_kept == 0:
        print(f"  !! Aucun point dans cette fenêtre — t_min={t.min():.3f}  t_max={t.max():.3f}")
        return None
    las[mask].write(out_path)
    print(f"  → {out_path.name}")
    return out_path

out_a = out_dir / "scan_7000_crop.las"
out_b = out_dir / "scan_8000_crop.las"

#out_a = crop_las_by_time(scan_a_path, *t_crop_a, out_a)
#out_b = crop_las_by_time(scan_b_path, *t_crop_b, out_b)

In [ ]:
# ==============================================================
# CELL 3 — LANCER LIMATCH
# ==============================================================
import sys
sys.path.insert(0, "/home/b085164/PDM_Romain_Defferrard/ESO-PDM")
from navtools_PDM.pipeline import run_limatch_api, get_repo_root

assert out_a is not None and out_b is not None, "Crop vide — ajuste t_crop_a / t_crop_b"

limatch_cfg = "/home/b085164/PDM_Romain_Defferrard/ESO-PDM/Patcher/submodules/limatch/configs/MLS.yml"
repo_root   = get_repo_root()

cfg_overrides = {"uncertainty_r": 30}

print(f"[limatch] cloud1 : {out_a.name}")
print(f"[limatch] cloud2 : {out_b.name}")

result = run_limatch_api(
    repo_root=repo_root,
    limatch_cfg_path=limatch_cfg,
    cloud1=out_a,
    cloud2=out_b,
    out_dir=out_dir,
    cfg_overrides=cfg_overrides,
)
print(f"[limatch] done — résultat : {result}")

[limatch] cloud1 : scan_7000_crop.las
[limatch] cloud2 : scan_8000_crop.las
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[15:53:31] INFO | Starting LiMatch pipeline...
[15:53:31] INFO |   └─ Will save data at: /media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/limatch_debug/7000_8000/
[15:53:31] INFO | Visualization set to False
[15:53:31] INFO | [0/7] Model setup
[15:53:31] INFO |   └─ Using CUDA for descriptor inference.
[15:53:31] INFO |   └─ Loading from: /home/b085164/PDM_Romain_Defferrard/ESO-PDM/Patcher/submodules/limatch/weights/model.pth


[limatch-api] overrides: {'uncertainty_r': 30}
[limatch-api] prj_folder=/media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/limatch_debug/7000_8000/


[15:53:31] INFO |   └─ Loaded nested checkpoint with key 'model'.
[15:53:31] INFO |   └─ Descriptor model successfully loaded and set to eval() mode.
[15:53:31] INFO | [1/7] Preprocessing
[15:53:31] INFO |   └─ Loading point clouds...
[15:53:43] INFO |   └─ Loaded laser vectors from LAS extra bytes: /media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/limatch_debug/7000_8000/scan_7000_crop.las
[15:53:50] INFO |   └─ Loaded laser vectors from LAS extra bytes: /media/b085164/LaCie/2026spring_RD/ECCR/georef_ALL_zone_3_outage/APX/georef_F2B/limatch_debug/7000_8000/scan_8000_crop.las
[15:53:51] INFO |   └─ No tiling... (all points kept)
[15:53:51] INFO |   └─ Shifting point clouds toward origin...
[15:53:51] INFO |       └─ [2533291.49 1155156.61     464.33] m...
[15:53:51] INFO |       └─ (Coordinates shifted back at export)
[15:53:54] INFO |   └─ Generated 1 valid tiles.
[15:53:54] INFO |   └─ Applying voxelization with size 0.025 m...
[15:54:00] INFO |       └─

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import folium
from folium.plugins import Draw
# ------------------------------------------------------------
# USER INPUT
# ------------------------------------------------------------
out2_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/base_v0/out/ODyN_GNSS_INS.out")
out3_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/APX/outage_2/out/outage_2_APX.out")
out4_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/APX/outage_2/F2B_z2/out/F2B_out2_apx.out")
label2 = "ODyN ref"
label3 = "ODyN outage APX"
label4 = "F2B"
t_start = None
t_end   = None
max_display_points = 5000
output_html = Path("/home/b085164/PDM_Romain_Defferrard/ESO-PDM/traj_comparison_apx_outage_2.html")

# ------------------------------------------------------------
# SBET dtype + helpers
# ------------------------------------------------------------
SBET_DTYPE = np.dtype([
    ("time",    np.float64), ("lat",  np.float64), ("lon",   np.float64),
    ("alt",     np.float64), ("vx",   np.float64), ("vy",    np.float64),
    ("vz",      np.float64), ("roll", np.float64), ("pitch", np.float64),
    ("heading", np.float64), ("wander",np.float64),("ax",    np.float64),
    ("ay",      np.float64), ("az",   np.float64), ("wx",    np.float64),
    ("wy",      np.float64), ("wz",   np.float64),
])

def load_sbet(path):
    return pd.DataFrame(np.fromfile(path, dtype=SBET_DTYPE))

def filter_time_window(df, t0, t1):
    mask = np.ones(len(df), dtype=bool)
    if t0 is not None: mask &= df["time"].to_numpy() >= t0
    if t1 is not None: mask &= df["time"].to_numpy() <= t1
    out = df.loc[mask].copy()
    if len(out) == 0:
        raise ValueError("No samples in selected time window.")
    return out

def decimate(arr, n):
    if len(arr) <= n: return arr
    return arr[np.linspace(0, len(arr)-1, n).astype(int)]

# ------------------------------------------------------------
# Load + filter
# ------------------------------------------------------------
df2 = filter_time_window(load_sbet(out2_path), t_start, t_end)
df3 = filter_time_window(load_sbet(out3_path), t_start, t_end)
df4 = filter_time_window(load_sbet(out4_path), t_start, t_end)

lat2 = np.degrees(decimate(df2["lat"].to_numpy(), max_display_points))
lon2 = np.degrees(decimate(df2["lon"].to_numpy(), max_display_points))
lat3 = np.degrees(decimate(df3["lat"].to_numpy(), max_display_points))
lon3 = np.degrees(decimate(df3["lon"].to_numpy(), max_display_points))
lat4 = np.degrees(decimate(df4["lat"].to_numpy(), max_display_points))
lon4 = np.degrees(decimate(df4["lon"].to_numpy(), max_display_points))

# ------------------------------------------------------------
# Folium map
# ------------------------------------------------------------
# ------------------------------------------------------------
# Folium map
# ------------------------------------------------------------
def add_traj_with_time(m, lats, lons, times, color, label, group):
    """Ajoute une trajectoire avec tooltip temps GPS sur chaque segment."""
    n = len(lats)
    for i in range(n - 1):
        folium.PolyLine(
            locations=[(lats[i], lons[i]), (lats[i+1], lons[i+1])],
            color=color, weight=2, opacity=0.9,
            tooltip=f"{label} | t = {times[i]:.3f} s",
        ).add_to(group)

center = [lat2.mean(), lon2.mean()]
m = folium.Map(
    location=center,
    zoom_start=14,
    max_zoom=22,  # au lieu de 19
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/">CARTO</a>',
    name="CartoDB Positron",
)

# Décimer aussi les temps
def decimate_times(df, n):
    arr = df["time"].to_numpy()
    if len(arr) <= n: return arr
    return arr[np.linspace(0, len(arr)-1, n).astype(int)]

t2 = decimate_times(df2, max_display_points)
t3 = decimate_times(df3, max_display_points)
t4 = decimate_times(df4, max_display_points)

fg2 = folium.FeatureGroup(name=label2, show=True).add_to(m)
add_traj_with_time(m, lat2, lon2, t2, "#e74c3c", label2, fg2)

fg3 = folium.FeatureGroup(name=label3, show=True).add_to(m)
add_traj_with_time(m, lat3, lon3, t3, "#2980b9", label3, fg3)

fg4 = folium.FeatureGroup(name=label4, show=True).add_to(m)
add_traj_with_time(m, lat4, lon4, t4, "#27ae60", label4, fg4)

legend_html = f"""
<div style="position:fixed;top:30px;right:30px;z-index:1000;
            background:white;padding:8px 12px;border-radius:6px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.3);font-size:13px;">
  <span style="color:#e74c3c;">&#9644;</span> {label2}<br>
  <span style="color:#2980b9;">&#9644;</span> {label3}<br>
  <span style="color:#27ae60;">&#9644;</span> {label4}<br>
  <span style="color:#e67e22;">&#9644;</span> {label5}
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

from folium import plugins
folium.plugins.MeasureControl(position="bottomright", primary_length_unit="meters", max_width=300).add_to(m)
m.get_root().html.add_child(folium.Element("""
<script>
  document.addEventListener("DOMContentLoaded", function() {
    var maps = Object.values(window).filter(v => v && v._leaflet_id);
    if (maps.length) L.control.scale({position:"bottomright", imperial:false, maxWidth:400}).addTo(maps[0]);
  });
</script>
"""))


Draw(
    export=True,           # bouton "Export" qui télécharge le GeoJSON
    filename="polygon.geojson",
    position="topleft",
    draw_options={
        "polyline": False,
        "rectangle": True,
        "polygon": True,
        "circle": False,
        "marker": False,
        "circlemarker": False,
    },
    edit_options={"edit": True},
).add_to(m)

folium.LayerControl().add_to(m)

m.save(str(output_html))
print(f"Saved: {output_html}")
print("-> Dessine un polygone dans la carte, clique 'Export' pour sauvegarder polygon.geojson")

Saved: /home/b085164/PDM_Romain_Defferrard/ESO-PDM/traj_comparison_apx_outage_2.html
-> Dessine un polygone dans la carte, clique 'Export' pour sauvegarder polygon.geojson


In [1]:
# ==============================================================
# CELL 1 — CONFIG
# ==============================================================
import numpy as np
import pandas as pd
import laspy
import json
from pathlib import Path
from shapely.geometry import shape, Point, box
from shapely.ops import unary_union

# polygone exporté depuis la carte folium
polygon_path = Path("/home/b085164/PDM_Romain_Defferrard/ESO-PDM/navtools_PDM/polygon.geojson")

# dossier contenant les merged_XXXX_VUX_PUCK.las
scans_dir = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/merged/ALL")

# SBET pour filtrer par trajectoire
sbet_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/APX/outage_2/F2B_z2/out/F2B_out2_apx.out")

# output
out_root = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/limatch_polygon")
out_root.mkdir(parents=True, exist_ok=True)

limatch_cfg = "/home/b085164/PDM_Romain_Defferrard/ESO-PDM/Patcher/submodules/limatch/configs/MLS_F2B_2.yml"
cfg_overrides = {"uncertainty_r": 40}

min_points_per_scan = 500  # ignorer les crops trop petits

In [3]:
# ==============================================================
# CELL 2 — LIRE LE POLYGONE ET LES SCANS
# ==============================================================
from pyproj import Transformer

# lire le polygone GeoJSON (en WGS84) et convertir en LV95
with open(polygon_path) as f:
    gj = json.load(f)

# prendre le premier feature
geom_wgs84 = shape(gj["features"][0]["geometry"])

# convertir les coordonnées du polygone en LV95
wgs2lv95 = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)

def poly_wgs84_to_lv95(geom):
    coords = list(geom.exterior.coords)
    coords_lv95 = [wgs2lv95.transform(lon, lat) for lon, lat in coords]
    from shapely.geometry import Polygon
    return Polygon(coords_lv95)

polygon_lv95 = poly_wgs84_to_lv95(geom_wgs84)
print(f"Polygone LV95 : bounds={polygon_lv95.bounds}")

# lister tous les fichiers merged
scan_files = sorted(scans_dir.glob("merged_*_VUX_PUCK.las"))
print(f"Fichiers trouvés : {len(scan_files)}")

Polygone LV95 : bounds=(2542576.310735904, 1157235.061245306, 2542739.2514480036, 1157409.8875815577)
Fichiers trouvés : 21


In [10]:
# ==============================================================
# CELL 3 — TROUVER LES SCANS QUI PASSENT DANS LE POLYGONE
# ==============================================================
from shapely.geometry import MultiPoint

scans_in_polygon = []

for sf in scan_files:
    las = laspy.read(sf)
    xy  = np.vstack((np.asarray(las.x), np.asarray(las.y))).T

    # test rapide : bbox overlap
    x_min, x_max = xy[:,0].min(), xy[:,0].max()
    y_min, y_max = xy[:,1].min(), xy[:,1].max()
    scan_bbox = box(x_min, y_min, x_max, y_max)

    if not polygon_lv95.intersects(scan_bbox):
        continue

    # test précis : points dans le polygone
    mask = np.array([polygon_lv95.contains(Point(x, y)) for x, y in xy[::20]])  # subsample x20
    n_in = int(mask.sum()) * 20  # estimation

    if n_in < min_points_per_scan:
        continue

    t = np.asarray(las.gps_time)
    t_in = t[::20][mask]
    if len(t_in) == 0:
        continue

    scans_in_polygon.append({
        "path":  sf,
        "name":  sf.stem,
        "t_min": float(t_in.min()),
        "t_max": float(t_in.max()),
        "n_pts": n_in,
    })
    print(f"  ✓ {sf.name} — ~{n_in} pts dans polygone  t=[{t_in.min():.2f}, {t_in.max():.2f}]")

print(f"\nTotal scans dans le polygone : {len(scans_in_polygon)}")

  ✓ merged_10000_VUX_PUCK.las — ~9453500 pts dans polygone  t=[305864.73, 305875.85]
  ✓ merged_1000_VUX_PUCK.las — ~6803400 pts dans polygone  t=[305697.13, 305705.93]
  ✓ merged_11000_VUX_PUCK.las — ~13533500 pts dans polygone  t=[305875.85, 305891.53]
  ✓ merged_12000_VUX_PUCK.las — ~6237040 pts dans polygone  t=[305891.53, 305899.08]
  ✓ merged_13000_VUX_PUCK.las — ~12126660 pts dans polygone  t=[305899.09, 305912.03]
  ✓ merged_14000_VUX_PUCK.las — ~5404240 pts dans polygone  t=[305912.03, 305918.51]
  ✓ merged_15000_VUX_PUCK.las — ~12719140 pts dans polygone  t=[305918.51, 305933.11]
  ✓ merged_16000_VUX_PUCK.las — ~9831020 pts dans polygone  t=[305933.11, 305942.73]
  ✓ merged_17000_VUX_PUCK.las — ~6232500 pts dans polygone  t=[305942.73, 305949.44]
  ✓ merged_18000_VUX_PUCK.las — ~12931540 pts dans polygone  t=[305949.44, 305965.53]
  ✓ merged_19000_VUX_PUCK.las — ~12386580 pts dans polygone  t=[306089.33, 306104.94]
  ✓ merged_2000_VUX_PUCK.las — ~22870420 pts dans polygone  t

In [11]:
# ==============================================================
# CELL 4 — CROP LES SCANS ET CONSTRUIRE LES PAIRES
# ==============================================================
import itertools

from shapely.vectorized import contains

def crop_las_to_polygon(las_path, polygon, out_path):
    las = laspy.read(las_path)
    x = np.asarray(las.x)
    y = np.asarray(las.y)
    
    # test vectorisé — des ordres de grandeur plus rapide
    mask = contains(polygon, x, y)
    
    n = int(mask.sum())
    if n == 0:
        return None, 0
    las[mask].write(out_path)
    return out_path, n

# crop tous les scans dans le polygone
cropped = {}
crop_dir = out_root / "cropped"
crop_dir.mkdir(exist_ok=True)

for scan in scans_in_polygon:
    out_path = crop_dir / f"{scan['name']}_crop.las"
    if out_path.exists():
        print(f"  (déjà cropé) {out_path.name}")
        cropped[scan["name"]] = out_path
        continue
    path, n = crop_las_to_polygon(scan["path"], polygon_lv95, out_path)
    if path is not None:
        print(f"  {scan['name']} → {n} pts → {out_path.name}")
        cropped[scan["name"]] = out_path
    else:
        print(f"  {scan['name']} → 0 pts — ignoré")

# toutes les paires possibles
names = list(cropped.keys())
pairs = list(itertools.combinations(names, 2))
print(f"\nPaires à matcher : {len(pairs)}")
for a, b in pairs:
    print(f"  {a} ↔ {b}")

  (déjà cropé) merged_10000_VUX_PUCK_crop.las
  (déjà cropé) merged_1000_VUX_PUCK_crop.las
  (déjà cropé) merged_11000_VUX_PUCK_crop.las
  (déjà cropé) merged_12000_VUX_PUCK_crop.las
  (déjà cropé) merged_13000_VUX_PUCK_crop.las
  (déjà cropé) merged_14000_VUX_PUCK_crop.las
  (déjà cropé) merged_15000_VUX_PUCK_crop.las
  (déjà cropé) merged_16000_VUX_PUCK_crop.las
  (déjà cropé) merged_17000_VUX_PUCK_crop.las
  (déjà cropé) merged_18000_VUX_PUCK_crop.las
  (déjà cropé) merged_19000_VUX_PUCK_crop.las
  (déjà cropé) merged_2000_VUX_PUCK_crop.las
  (déjà cropé) merged_3000_VUX_PUCK_crop.las
  (déjà cropé) merged_4000_VUX_PUCK_crop.las
  (déjà cropé) merged_5000_VUX_PUCK_crop.las
  (déjà cropé) merged_6000_VUX_PUCK_crop.las
  (déjà cropé) merged_7000_VUX_PUCK_crop.las
  (déjà cropé) merged_8000_VUX_PUCK_crop.las
  (déjà cropé) merged_9000_VUX_PUCK_crop.las

Paires à matcher : 171
  merged_10000_VUX_PUCK ↔ merged_1000_VUX_PUCK
  merged_10000_VUX_PUCK ↔ merged_11000_VUX_PUCK
  merged_10000_V

In [ ]:
# ==============================================================
# CELL 5 — MANIFEST DES PAIRES
# ==============================================================
manifest_path = out_root / "pairs_manifest.csv"

rows = []
for a_name, b_name in pairs:
    rows.append({
        "cloud1": str(cropped[a_name]),
        "cloud2": str(cropped[b_name]),
        "pair_name": f"{a_name}__{b_name}",
        "out_dir": str(out_root / f"{a_name}__{b_name}"),
        "status": "pending",
    })

df_manifest = pd.DataFrame(rows)
df_manifest.to_csv(manifest_path, index=False)
print(f"Manifest écrit : {manifest_path}")
print(df_manifest[["pair_name", "status"]].to_string())

In [ ]:
# ==============================================================
# CELL 5 — LANCER LIMATCH SUR TOUTES LES PAIRES
# ==============================================================
import sys
sys.path.insert(0, "/home/b085164/PDM_Romain_Defferrard/ESO-PDM")
from navtools_PDM.pipeline import run_limatch_api, get_repo_root

repo_root = get_repo_root()
results = {}

for a_name, b_name in pairs:
    pair_name = f"{a_name}__{b_name}"
    pair_out  = out_root / pair_name
    pair_out.mkdir(parents=True, exist_ok=True)

    cloud1 = cropped[a_name]
    cloud2 = cropped[b_name]

    print(f"\n[limatch] {a_name} ↔ {b_name}")
    try:
        result = run_limatch_api(
            repo_root=repo_root,
            limatch_cfg_path=limatch_cfg,
            cloud1=cloud1,
            cloud2=cloud2,
            out_dir=pair_out,
            cfg_overrides=cfg_overrides,
        )
        results[pair_name] = {"status": "ok", "result": result}
        print(f"  ✓ done")
    except Exception as e:
        results[pair_name] = {"status": "fail", "error": str(e)}
        print(f"  ✗ FAIL : {e}")

# résumé
print(f"\n{'='*50}")
print(f"Résumé : {sum(1 for v in results.values() if v['status']=='ok')}/{len(pairs)} paires réussies")
for name, r in results.items():
    status = "✓" if r["status"] == "ok" else "✗"
    print(f"  {status} {name}")


[limatch] merged_10000_VUX_PUCK ↔ merged_1000_VUX_PUCK
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[17:08:52] INFO | Starting LiMatch pipeline...
[17:08:52] INFO |   └─ Will save data at: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/limatch_polygon/merged_10000_VUX_PUCK__merged_1000_VUX_PUCK/
[17:08:52] INFO | Visualization set to False
[17:08:52] INFO | [0/7] Model setup
[17:08:52] INFO |   └─ Using CUDA for descriptor inference.
[17:08:52] INFO |   └─ Loading from: /home/b085164/PDM_Romain_Defferrard/ESO-PDM/Patcher/submodules/limatch/weights/model.pth


[limatch-api] overrides: {'uncertainty_r': 40}
[limatch-api] prj_folder=/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/limatch_polygon/merged_10000_VUX_PUCK__merged_1000_VUX_PUCK/


[17:08:52] INFO |   └─ Loaded nested checkpoint with key 'model'.
[17:08:52] INFO |   └─ Descriptor model successfully loaded and set to eval() mode.
[17:08:52] INFO | [1/7] Preprocessing
[17:08:52] INFO |   └─ Loading point clouds...
[17:08:56] INFO |   └─ Loaded laser vectors from LAS extra bytes: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/limatch_polygon/cropped/merged_10000_VUX_PUCK_crop.las
[17:09:01] INFO |   └─ Loaded laser vectors from LAS extra bytes: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_2/APX/georef_F2B/limatch_polygon/cropped/merged_1000_VUX_PUCK_crop.las
[17:09:02] INFO |   └─ No tiling... (all points kept)
[17:09:02] INFO |   └─ Shifting point clouds toward origin...
[17:09:02] INFO |       └─ [2542668.27 1157385.69     907.8 ] m...
[17:09:02] INFO |       └─ (Coordinates shifted back at export)
[17:09:06] INFO |   └─ Generated 1 valid tiles.
[17:09:06] INFO |   └─ Applying voxelization with size 0.025 m...
[17:0